# Statistical Significance Testing — Spam Detection Models

## Implements two formal tests required for the IEEE paper:
1. **Dietterich's 5×2 CV Paired t-test** — pairwise comparison of top model pairs
2. **Bootstrap 95% Confidence Intervals** — accuracy, F1, AUC for all key models

### Model pairs tested:
- **A vs B**: XGB Optuna (top-30, eng.) vs RF Standardised Baseline
- **A vs C**: XGB Optuna (top-30, eng.) vs XGB Default (orig.)
- **A vs D**: XGB Optuna (top-30, eng.) vs Stacking Ensemble
- **A vs E**: XGB Optuna (full 64) vs XGB Default (orig.)

**Reference:** Dietterich, T.G. (1998). Approximate statistical tests for comparing supervised  
classification learning algorithms. *Neural Computation*, 10(7), 1895–1923.

> All random seeds match exactly those used in paper_experiments and advanced_experiments notebooks.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from scipy import stats

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

FIG_DIR = 'significance_figures'
os.makedirs(FIG_DIR, exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


## 1. Data Loading & Feature Engineering
Replicates the exact pipeline from `advanced_experiments_executed.ipynb`.

In [2]:
# Load raw spambase (use spambase.data — the original, un-balanced dataset)
spambase_cols = (
    [f'word_freq_{w}' for w in [
        'make','address','all','3d','our','over','remove','internet',
        'order','mail','receive','will','people','report','addresses',
        'free','business','email','you','credit','your','font','000',
        'money','hp','hpl','george','650','lab','labs','telnet','857',
        'data','415','85','technology','1999','parts','pm','direct',
        'cs','meeting','original','project','re','edu','table','conference'
    ]] +
    ['char_freq_semicolon','char_freq_paren','char_freq_bracket',
     'char_freq_exclaim','char_freq_dollar','char_freq_hash'] +
    ['capital_run_length_average','capital_run_length_longest',
     'capital_run_length_total'] +
    ['spam']
)

df_raw = pd.read_csv('spambase.data', header=None, names=spambase_cols)
print(f'Raw dataset shape: {df_raw.shape}')
print(f'Class distribution — Ham: {(df_raw.spam==0).sum()}, Spam: {(df_raw.spam==1).sum()}')

ORIG_FEATURES = spambase_cols[:-1]
X_raw = df_raw[ORIG_FEATURES].values
y_raw = df_raw['spam'].values

Raw dataset shape: (4601, 58)
Class distribution — Ham: 2788, Spam: 1813


In [3]:
def engineer_features(df):
    """Replicate the 7 interaction features from Section III-C of the paper."""
    d = df.copy()
    eps = 1e-9

    # log1p transform on all 57 features (Strategy B)
    for col in ORIG_FEATURES:
        d[col] = np.log1p(d[col])

    # f1: capital intensity
    d['feat_capital_intensity'] = (
        d['capital_run_length_total'] * d['capital_run_length_longest']
    ) / (d['capital_run_length_average'] + eps)

    # f2: spam signal score
    spam_words = ['make','address','all','3d','our','over','remove','internet',
                  'order','mail','receive','will','people','report','addresses',
                  'free','business','email','you','credit','your','font','000','money']
    spam_word_cols = [f'word_freq_{w}' for w in spam_words]
    d['feat_spam_signal'] = d[spam_word_cols].sum(axis=1) + d['char_freq_exclaim'] + d['char_freq_dollar']

    # f3: ham signal score
    d['feat_ham_signal'] = (
        d['word_freq_hp'] + d['word_freq_hpl'] + d['word_freq_george'] +
        d['word_freq_meeting'] + d['word_freq_re']
    )

    # f4: spam-to-ham ratio
    d['feat_spam_ham_ratio'] = d['feat_spam_signal'] / (d['feat_ham_signal'] + 0.01)

    # f5: capital-exclaim cross
    d['feat_cap_exclaim'] = (
        np.log1p(df['capital_run_length_total']) * np.log1p(df['char_freq_exclaim'])
    )

    # f6: capital-dollar cross
    d['feat_cap_dollar'] = (
        np.log1p(df['capital_run_length_total']) * np.log1p(df['char_freq_dollar'])
    )

    # f7: word diversity
    word_freq_cols = [c for c in ORIG_FEATURES if c.startswith('word_freq')]
    d['feat_word_diversity'] = (df[word_freq_cols] > 0).sum(axis=1)

    return d


df_eng = engineer_features(df_raw[ORIG_FEATURES + ['spam']].copy())
ENG_FEATURES = [c for c in df_eng.columns if c != 'spam']
X_eng = df_eng[ENG_FEATURES].values
y_eng = df_raw['spam'].values

print(f'Engineered feature set: {X_eng.shape[1]} features')
print(f'Original feature set  : {X_raw.shape[1]} features')

Engineered feature set: 64 features
Original feature set  : 57 features


## 2. Define Models

Best Optuna parameters from `advanced_experiments_executed.ipynb`  
(350 estimators, depth 8, lr=0.039, subsample=0.941, colsample=0.629)

In [4]:
# ── Best Optuna-XGB params (from paper, Section III-E) ──────────────────────
OPTUNA_XGB_PARAMS = dict(
    n_estimators=350,
    max_depth=8,
    learning_rate=0.039,
    subsample=0.941,
    colsample_bytree=0.629,
    min_child_weight=1,
    reg_alpha=1e-4,
    reg_lambda=1e-4,
    gamma=0,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    verbosity=0,
)

def make_models():
    """Return a fresh dict of (name -> pipeline) for each comparison."""
    xgb_default_orig = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', XGBClassifier(
            eval_metric='logloss', use_label_encoder=False,
            random_state=RANDOM_STATE, verbosity=0
        ))
    ])

    rf_baseline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE
        ))
    ])

    xgb_optuna_eng = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', XGBClassifier(**OPTUNA_XGB_PARAMS))
    ])

    # Stacking: XGB + RF + SVM(RBF) + GB -> LR meta
    base_learners = [
        ('xgb', XGBClassifier(
            eval_metric='logloss', use_label_encoder=False,
            random_state=RANDOM_STATE, verbosity=0
        )),
        ('rf',  RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
        ('svm', SVC(C=10, kernel='rbf', probability=True, random_state=RANDOM_STATE)),
        ('gb',  GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.1,
            random_state=RANDOM_STATE
        )),
    ]
    stacking = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', StackingClassifier(
            estimators=base_learners,
            final_estimator=LogisticRegression(C=1.0, random_state=RANDOM_STATE),
            cv=5, passthrough=False, n_jobs=-1
        ))
    ])

    return {
        'XGB_Default_Orig':  (xgb_default_orig,  X_raw),
        'RF_Baseline':       (rf_baseline,        X_raw),
        'XGB_Optuna_Eng64':  (xgb_optuna_eng,     X_eng),
        'Stacking_Orig':     (stacking,           X_raw),
    }

print('Model factory ready.')
print('Models:', list(make_models().keys()))

Model factory ready.
Models: ['XGB_Default_Orig', 'RF_Baseline', 'XGB_Optuna_Eng64', 'Stacking_Orig']


## 3. Dietterich's 5×2 CV Paired t-test

**Algorithm (Dietterich 1998):**
1. Repeat 5 times (i = 1..5):
   - Split data randomly 50/50 (stratified)
   - Train A & B on fold-1, test on fold-2  → diff $p_i^{(1)} = err_A - err_B$
   - Train A & B on fold-2, test on fold-1  → diff $p_i^{(2)} = err_A - err_B$
   - $\bar{p}_i = (p_i^{(1)} + p_i^{(2)}) / 2$
   - $s_i^2 = (p_i^{(1)} - \bar{p}_i)^2 + (p_i^{(2)} - \bar{p}_i)^2$
2. $t = p_1^{(1)} / \sqrt{\frac{1}{5} \sum_{i=1}^{5} s_i^2}$
3. t follows t-distribution with 5 df → two-tailed p-value

**Pairs tested:**
- XGB Optuna (Eng-64) vs RF Baseline
- XGB Optuna (Eng-64) vs XGB Default (orig.)
- XGB Optuna (Eng-64) vs Stacking
- RF Baseline vs XGB Default (orig.)

In [5]:
def dietterich_5x2cv_ttest(model_A, X_A, model_B, X_B, y, n_splits=5, random_state=RANDOM_STATE):
    """
    Dietterich (1998) 5x2 CV paired t-test.
    Both models may use different feature matrices (X_A, X_B),
    but y must be shared.
    Returns: t_stat, p_value, list of per-round diffs
    """
    import copy
    rng = np.random.RandomState(random_state)
    s_sq_list = []
    p1_first = None  # p_1^{(1)} for numerator
    all_diffs = []

    for i in range(n_splits):
        # Create a fresh 50/50 stratified split
        seed_i = rng.randint(0, 100000)
        skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=seed_i)
        folds = list(skf.split(X_A, y))  # [(train_idx, test_idx), (train_idx, test_idx)]

        # fold (1): train on folds[0][0], test on folds[0][1]
        tr1, te1 = folds[0]
        # fold (2): train on folds[1][0], test on folds[1][1]
        tr2, te2 = folds[1]

        # When models use different feature matrices we need to index both
        Xa_tr1, Xa_te1 = X_A[tr1], X_A[te1]
        Xa_tr2, Xa_te2 = X_A[tr2], X_A[te2]
        Xb_tr1, Xb_te1 = X_B[tr1], X_B[te1]
        Xb_tr2, Xb_te2 = X_B[tr2], X_B[te2]
        y_tr1, y_te1 = y[tr1], y[te1]
        y_tr2, y_te2 = y[tr2], y[te2]

        # Clone models for each fold
        mA1 = copy.deepcopy(model_A); mA2 = copy.deepcopy(model_A)
        mB1 = copy.deepcopy(model_B); mB2 = copy.deepcopy(model_B)

        # Fold 1 (train on tr1, test on te1)
        mA1.fit(Xa_tr1, y_tr1); mB1.fit(Xb_tr1, y_tr1)
        err_A1 = 1.0 - accuracy_score(y_te1, mA1.predict(Xa_te1))
        err_B1 = 1.0 - accuracy_score(y_te1, mB1.predict(Xb_te1))
        p1 = err_A1 - err_B1

        # Fold 2 (train on tr2, test on te2)
        mA2.fit(Xa_tr2, y_tr2); mB2.fit(Xb_tr2, y_tr2)
        err_A2 = 1.0 - accuracy_score(y_te2, mA2.predict(Xa_te2))
        err_B2 = 1.0 - accuracy_score(y_te2, mB2.predict(Xb_te2))
        p2 = err_A2 - err_B2

        p_bar = (p1 + p2) / 2.0
        s_sq = (p1 - p_bar) ** 2 + (p2 - p_bar) ** 2
        s_sq_list.append(s_sq)
        all_diffs.extend([p1, p2])

        if i == 0:
            p1_first = p1  # numerator uses p_1^{(1)}

    variance_est = (1.0 / n_splits) * np.sum(s_sq_list)
    t_stat = p1_first / np.sqrt(variance_est + 1e-15)
    p_value = 2.0 * stats.t.sf(np.abs(t_stat), df=n_splits)

    return t_stat, p_value, all_diffs


print('5x2 CV t-test function defined.')

5x2 CV t-test function defined.


In [6]:
import copy, time

# Define pairs: (name_A, name_B)
PAIRS = [
    ('XGB_Optuna_Eng64', 'RF_Baseline'),
    ('XGB_Optuna_Eng64', 'XGB_Default_Orig'),
    ('XGB_Optuna_Eng64', 'Stacking_Orig'),
    ('RF_Baseline',      'XGB_Default_Orig'),
]

sig_results = []

for name_A, name_B in PAIRS:
    models = make_models()
    model_A, X_A = models[name_A]
    model_B, X_B = models[name_B]

    print(f'Running 5x2 CV: {name_A}  vs  {name_B} ...', end=' ', flush=True)
    t0 = time.time()

    t_stat, p_val, diffs = dietterich_5x2cv_ttest(
        model_A, X_A, model_B, X_B, y_raw
    )

    elapsed = time.time() - t0
    sig = 'YES (p<0.05)' if p_val < 0.05 else 'NO (p>=0.05)'
    print(f't={t_stat:.4f}, p={p_val:.4f}  [{sig}]  ({elapsed:.1f}s)')

    sig_results.append({
        'Model A': name_A,
        'Model B': name_B,
        't-stat': round(t_stat, 4),
        'p-value': round(p_val, 4),
        'Significant (α=0.05)': 'Yes' if p_val < 0.05 else 'No',
        'Mean diff (err_A - err_B)': round(np.mean(diffs), 5),
    })

sig_df = pd.DataFrame(sig_results)
print('\n=== 5×2 CV Paired t-test Results ===')
print(sig_df.to_string(index=False))

Running 5x2 CV: XGB_Optuna_Eng64  vs  RF_Baseline ... 

t=-1.6064, p=0.1691  [NO (p>=0.05)]  (9.6s)
Running 5x2 CV: XGB_Optuna_Eng64  vs  XGB_Default_Orig ... 

t=-1.4711, p=0.2012  [NO (p>=0.05)]  (7.3s)
Running 5x2 CV: XGB_Optuna_Eng64  vs  Stacking_Orig ... 

t=-0.0858, p=0.9349  [NO (p>=0.05)]  (31.5s)
Running 5x2 CV: RF_Baseline  vs  XGB_Default_Orig ... 

t=0.4081, p=0.7001  [NO (p>=0.05)]  (4.8s)

=== 5×2 CV Paired t-test Results ===
         Model A          Model B  t-stat  p-value Significant (α=0.05)  Mean diff (err_A - err_B)
XGB_Optuna_Eng64      RF_Baseline -1.6064   0.1691                   No                   -0.00313
XGB_Optuna_Eng64 XGB_Default_Orig -1.4711   0.2012                   No                   -0.00156
XGB_Optuna_Eng64    Stacking_Orig -0.0858   0.9349                   No                    0.00096
     RF_Baseline XGB_Default_Orig  0.4081   0.7001                   No                    0.00156


In [7]:
sig_df.to_csv('significance_5x2cv_results.csv', index=False)
print('Saved significance_5x2cv_results.csv')

Saved significance_5x2cv_results.csv


## 4. Bootstrap 95% Confidence Intervals

For each model:
- Train on the full 80% train split (stratified, random_state=0)
- Draw **B=2000** bootstrap samples from the 20% test set  
- Compute accuracy, F1, AUC for each resample
- Report mean ± std and the [2.5th, 97.5th] percentile CI

In [8]:
def bootstrap_ci(y_true, y_pred, y_prob, B=2000, alpha=0.05, random_state=RANDOM_STATE):
    """
    Percentile bootstrap CI for accuracy, macro-F1, and AUC-ROC.
    Returns dict with mean, std, lower, upper for each metric.
    """
    rng = np.random.RandomState(random_state)
    n = len(y_true)
    acc_b, f1_b, auc_b = [], [], []

    for _ in range(B):
        idx = rng.randint(0, n, size=n)
        yt, yp, ypr = y_true[idx], y_pred[idx], y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue  # skip degenerate bootstrap samples
        acc_b.append(accuracy_score(yt, yp))
        f1_b.append(f1_score(yt, yp, zero_division=0))
        auc_b.append(roc_auc_score(yt, ypr))

    lo, hi = alpha / 2 * 100, (1 - alpha / 2) * 100

    def ci(arr):
        arr = np.array(arr)
        return {
            'mean': np.mean(arr),
            'std':  np.std(arr),
            'lower': np.percentile(arr, lo),
            'upper': np.percentile(arr, hi),
        }

    return {'accuracy': ci(acc_b), 'f1': ci(f1_b), 'auc': ci(auc_b)}


# ── Build test predictions for each model on the NATURAL (unbalanced) split ──
# Matches the main paper experiments: random_state=0, 80/20

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=0, stratify=y_raw
)
X_train_eng, X_test_eng, _, _ = train_test_split(
    X_eng, y_raw, test_size=0.2, random_state=0, stratify=y_raw
)

print(f'Train: {X_train_raw.shape[0]}, Test: {X_test_raw.shape[0]}')
print(f'Test spam rate: {y_test.mean():.3f}')

Train: 3680, Test: 921
Test spam rate: 0.394


In [9]:
BOOTSTRAP_MODELS = {
    'XGB Default (orig.)': (
        Pipeline([
            ('scaler', StandardScaler()),
            ('clf', XGBClassifier(
                eval_metric='logloss', use_label_encoder=False,
                random_state=RANDOM_STATE, verbosity=0
            ))
        ]),
        X_train_raw, X_test_raw
    ),
    'RF Baseline': (
        Pipeline([
            ('scaler', StandardScaler()),
            ('clf', RandomForestClassifier(
                n_estimators=200, random_state=RANDOM_STATE
            ))
        ]),
        X_train_raw, X_test_raw
    ),
    'XGB Optuna (eng. 64)': (
        Pipeline([
            ('scaler', StandardScaler()),
            ('clf', XGBClassifier(**OPTUNA_XGB_PARAMS))
        ]),
        X_train_eng, X_test_eng
    ),
    'Stacking (XGB+RF+SVM+GB)': (
        Pipeline([
            ('scaler', StandardScaler()),
            ('clf', StackingClassifier(
                estimators=[
                    ('xgb', XGBClassifier(
                        eval_metric='logloss', use_label_encoder=False,
                        random_state=RANDOM_STATE, verbosity=0
                    )),
                    ('rf',  RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
                    ('svm', SVC(C=10, kernel='rbf', probability=True, random_state=RANDOM_STATE)),
                    ('gb',  GradientBoostingClassifier(
                        n_estimators=200, max_depth=3, learning_rate=0.1,
                        random_state=RANDOM_STATE
                    )),
                ],
                final_estimator=LogisticRegression(C=1.0, random_state=RANDOM_STATE),
                cv=5, passthrough=False, n_jobs=-1
            ))
        ]),
        X_train_raw, X_test_raw
    ),
}

boot_results = {}

for name, (pipe, Xtr, Xte) in BOOTSTRAP_MODELS.items():
    print(f'Fitting {name} ...', end=' ', flush=True)
    t0 = time.time()
    pipe.fit(Xtr, y_train)
    y_pred = pipe.predict(Xte)
    y_prob = pipe.predict_proba(Xte)[:, 1]
    ci = bootstrap_ci(y_test, y_pred, y_prob, B=2000)
    boot_results[name] = ci
    elapsed = time.time() - t0
    print(f'Acc={ci["accuracy"]["mean"]:.4f} '
          f'[{ci["accuracy"]["lower"]:.4f}, {ci["accuracy"]["upper"]:.4f}]  '
          f'({elapsed:.1f}s)')

print('\nBootstrap CIs computed.')

Fitting XGB Default (orig.) ... 

Acc=0.9576 [0.9435, 0.9696]  (2.1s)
Fitting RF Baseline ... 

Acc=0.9555 [0.9414, 0.9685]  (2.5s)
Fitting XGB Optuna (eng. 64) ... 

Acc=0.9566 [0.9435, 0.9696]  (2.6s)
Fitting Stacking (XGB+RF+SVM+GB) ... 

Acc=0.9619 [0.9490, 0.9739]  (5.5s)

Bootstrap CIs computed.


In [10]:
rows = []
for name, ci in boot_results.items():
    rows.append({
        'Model': name,
        'Acc mean':  round(ci['accuracy']['mean'], 4),
        'Acc 95% CI': f"[{ci['accuracy']['lower']:.4f}, {ci['accuracy']['upper']:.4f}]",
        'F1 mean':   round(ci['f1']['mean'], 4),
        'F1 95% CI': f"[{ci['f1']['lower']:.4f}, {ci['f1']['upper']:.4f}]",
        'AUC mean':  round(ci['auc']['mean'], 4),
        'AUC 95% CI': f"[{ci['auc']['lower']:.4f}, {ci['auc']['upper']:.4f}]",
    })

boot_df = pd.DataFrame(rows)
print('=== Bootstrap 95% Confidence Intervals (B=2000) ===')
print(boot_df.to_string(index=False))

boot_df.to_csv('significance_bootstrap_ci.csv', index=False)
print('\nSaved significance_bootstrap_ci.csv')

=== Bootstrap 95% Confidence Intervals (B=2000) ===
                   Model  Acc mean       Acc 95% CI  F1 mean        F1 95% CI  AUC mean       AUC 95% CI
     XGB Default (orig.)    0.9576 [0.9435, 0.9696]   0.9465 [0.9287, 0.9624]    0.9887 [0.9811, 0.9948]
             RF Baseline    0.9555 [0.9414, 0.9685]   0.9431 [0.9238, 0.9599]    0.9888 [0.9812, 0.9951]
    XGB Optuna (eng. 64)    0.9566 [0.9435, 0.9696]   0.9450 [0.9277, 0.9612]    0.9892 [0.9817, 0.9950]
Stacking (XGB+RF+SVM+GB)    0.9619 [0.9490, 0.9739]   0.9520 [0.9345, 0.9677]    0.9897 [0.9820, 0.9956]

Saved significance_bootstrap_ci.csv


## 5. Figures

Two publication-ready figures:
1. **Bootstrap CI plot** — error-bar chart of accuracy 95% CIs per model
2. **t-test summary** — heatmap-style table of pairwise significance

In [11]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['accuracy', 'f1', 'auc']
labels  = ['Accuracy', 'F1-Score', 'AUC-ROC']
model_names = list(boot_results.keys())
x_pos = np.arange(len(model_names))

colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

for ax, metric, label in zip(axes, metrics, labels):
    means  = [boot_results[m][metric]['mean']  for m in model_names]
    lowers = [boot_results[m][metric]['lower'] for m in model_names]
    uppers = [boot_results[m][metric]['upper'] for m in model_names]
    errs_lo = [m - l for m, l in zip(means, lowers)]
    errs_hi = [u - m for m, u in zip(means, uppers)]

    ax.bar(x_pos, means, color=colors, alpha=0.8, zorder=2)
    ax.errorbar(
        x_pos, means,
        yerr=[errs_lo, errs_hi],
        fmt='none', color='black', capsize=6, capthick=2, linewidth=2, zorder=3
    )
    ax.set_xticks(x_pos)
    ax.set_xticklabels(
        [n.replace(' ', '\n') for n in model_names],
        fontsize=8
    )
    ax.set_ylabel(label)
    ax.set_title(f'{label} with 95% Bootstrap CI')
    ax.set_ylim(min(lowers) * 0.995, 1.005)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.3f}'))
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    for xi, (m, lo, hi) in enumerate(zip(means, lowers, uppers)):
        ax.annotate(
            f'{m:.4f}\n[{lo:.4f},{hi:.4f}]',
            xy=(xi, m), xytext=(0, 8), textcoords='offset points',
            ha='center', va='bottom', fontsize=6.5
        )

plt.suptitle('Bootstrap 95% Confidence Intervals — Spam Detection Models (B=2000)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/bootstrap_confidence_intervals.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


In [12]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

col_labels = ['Model A', 'Model B', 't-stat', 'p-value', 'Significant\n(α=0.05)', 'Mean diff\n(err_A − err_B)']
table_data = []
for _, row in sig_df.iterrows():
    table_data.append([
        row['Model A'].replace('_', '\n'),
        row['Model B'].replace('_', '\n'),
        f"{row['t-stat']:.4f}",
        f"{row['p-value']:.4f}",
        row['Significant (α=0.05)'],
        f"{row['Mean diff (err_A - err_B)']:.5f}",
    ])

tbl = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    loc='center',
    cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 2.0)

for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#1565C0')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        sig_val = table_data[r - 1][4]
        cell.set_facecolor('#C8E6C9' if sig_val == 'Yes' else '#FFCCBC')

ax.set_title('Dietterich 5×2 CV Paired t-test Results', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/5x2cv_ttest_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


## 6. Print Paper-Ready Sentences

Copy-paste ready text blocks for the IEEE paper.

In [13]:
print('=' * 70)
print('PAPER-READY STATISTICAL SIGNIFICANCE TEXT')
print('=' * 70)
print()

# Bootstrap block
print('--- Bootstrap CIs ---')
for name, ci in boot_results.items():
    a = ci['accuracy']
    f = ci['f1']
    u = ci['auc']
    print(f'{name}:')
    print(f'  Accuracy: {a["mean"]:.4f} (95% CI [{a["lower"]:.4f}, {a["upper"]:.4f}])')
    print(f'  F1-Score: {f["mean"]:.4f} (95% CI [{f["lower"]:.4f}, {f["upper"]:.4f}])')
    print(f'  AUC-ROC:  {u["mean"]:.4f} (95% CI [{u["lower"]:.4f}, {u["upper"]:.4f}])')
    print()

print('--- 5×2 CV Paired t-test ---')
for _, row in sig_df.iterrows():
    sig_str = 'statistically significant' if row['Significant (α=0.05)'] == 'Yes' else 'NOT statistically significant'
    print(f"{row['Model A']} vs {row['Model B']}: "
          f"t={row['t-stat']:.4f}, p={row['p-value']:.4f} — {sig_str} at α=0.05")

print()
print('=' * 70)

PAPER-READY STATISTICAL SIGNIFICANCE TEXT

--- Bootstrap CIs ---
XGB Default (orig.):
  Accuracy: 0.9576 (95% CI [0.9435, 0.9696])
  F1-Score: 0.9465 (95% CI [0.9287, 0.9624])
  AUC-ROC:  0.9887 (95% CI [0.9811, 0.9948])

RF Baseline:
  Accuracy: 0.9555 (95% CI [0.9414, 0.9685])
  F1-Score: 0.9431 (95% CI [0.9238, 0.9599])
  AUC-ROC:  0.9888 (95% CI [0.9812, 0.9951])

XGB Optuna (eng. 64):
  Accuracy: 0.9566 (95% CI [0.9435, 0.9696])
  F1-Score: 0.9450 (95% CI [0.9277, 0.9612])
  AUC-ROC:  0.9892 (95% CI [0.9817, 0.9950])

Stacking (XGB+RF+SVM+GB):
  Accuracy: 0.9619 (95% CI [0.9490, 0.9739])
  F1-Score: 0.9520 (95% CI [0.9345, 0.9677])
  AUC-ROC:  0.9897 (95% CI [0.9820, 0.9956])

--- 5×2 CV Paired t-test ---
XGB_Optuna_Eng64 vs RF_Baseline: t=-1.6064, p=0.1691 — NOT statistically significant at α=0.05
XGB_Optuna_Eng64 vs XGB_Default_Orig: t=-1.4711, p=0.2012 — NOT statistically significant at α=0.05
XGB_Optuna_Eng64 vs Stacking_Orig: t=-0.0858, p=0.9349 — NOT statistically significan